# Agentic Workflow: Routing Pattern

## Setup and Client Initialization

We read the `GEMINI_API_KEY` from the `.env` file in the parent directory and initialize the OpenAI client pointing to the Gemini API's OpenAI-compatible endpoint.

In [1]:
import os
import json
import re
from dotenv import load_dotenv
from openai import OpenAI

# Load environment variables from .env in the parent directory
dir_path = os.path.dirname(__file__) if "__file__" in globals() else "."
dotenv_path = os.path.abspath(os.path.join(dir_path, "..", ".env"))
load_dotenv(dotenv_path=dotenv_path)

# Retrieve the API key from environment variables
api_key = os.getenv("GEMINI_API_KEY")

if not api_key or api_key == "your_actual_api_key_here":
    raise ValueError(
        "Please set your GEMINI_API_KEY in the .env file in the root of the project.\n"
        "You can get a free key from Google AI Studio: https://aistudio.google.com/"
    )

# Initialize the OpenAI client pointing to Gemini's OpenAI-compatible API endpoint
client = OpenAI(
    api_key=api_key,
    base_url="https://generativelanguage.googleapis.com/v1beta/openai/"
)

# We use the free gemini-3.1-flash-lite model via the OpenAI compatible endpoint.
model_name = "gemini-3.1-flash-lite"
print("Gemini API client initialized successfully!")

Gemini API client initialized successfully!


## Database Initialization

We load database records from `database.json`. This simulates checking our system records for invoices and orders.

In [2]:
db_path = os.path.join(dir_path, "database.json")

def load_db():
    if not os.path.exists(db_path):
        raise FileNotFoundError(f"Database file not found at {db_path}")
    with open(db_path, "r") as f:
        return json.load(f)

# Verify database load
db = load_db()
print(f"Database loaded! Found {len(db['invoices'])} invoice records and {len(db['orders'])} order records.")

Database loaded! Found 3 invoice records and 3 order records.


## Base Completion Function

A helper function to get chat completions from the Gemini model using specific instructions.

In [3]:
def get_completion(prompt, system_instruction="You are a helpful customer support assistant."):
    try:
        response = client.chat.completions.create(
            model=model_name,
            messages=[
                {"role": "system", "content": system_instruction},
                {"role": "user", "content": prompt}
            ],
            temperature=0,
        )
        return response.choices[0].message.content.strip()
    except Exception as e:
        print(f"An error occurred: {e}")
        return None

## 1. The Router AI (Classifier)

The router reviews the user request and classifies it into one of the three specialist categories: `BILLING`, `TECH_SUPPORT`, or `RETURNS`.

In [4]:
ROUTER_SYSTEM_INSTRUCTION = (
    "You are a triage router for customer support. Analyze the customer's request "
    "and classify it into exactly one of these categories:\n"
    "- BILLING: for questions about bills, payments, invoices, or charges.\n"
    "- TECH_SUPPORT: for questions about product setup, troubleshooting, errors, crashes, or how-to guides.\n"
    "- RETURNS: for questions about returning a product, return status, or shipping labels.\n\n"
    "Output ONLY the category name: BILLING, TECH_SUPPORT, or RETURNS. Do not output any punctuation, "
    "introductory text, or explanation. Output must be exactly one of those three words."
)

def route_query(user_query):
    raw_route = get_completion(user_query, system_instruction=ROUTER_SYSTEM_INSTRUCTION)
    if not raw_route:
        return "UNKNOWN"
    
    # Standardize output
    cleaned_route = raw_route.strip().upper()
    for category in ["BILLING", "TECH_SUPPORT", "RETURNS"]:
        if category in cleaned_route:
            return category
            
    return "UNKNOWN"

## 2. Billing Specialist Agent

Handles billing queries. It attempts to extract an invoice ID (format `INV-XXXX`), looks it up in the JSON database, and provides a customized status update.

In [5]:
def handle_billing(query):
    # 1. Try to extract invoice ID using regex
    invoice_match = re.search(r"INV-\d{4}", query, re.IGNORECASE)
    
    if not invoice_match:
        # Use LLM fallback for fuzzy extraction
        extraction_prompt = f"Extract the invoice ID (format INV-XXXX) from this query. Return only the ID. If not found, reply 'NONE'.\nQuery: {query}"
        extracted = get_completion(extraction_prompt, "You are an invoice ID extraction assistant.")
        invoice_match = re.search(r"INV-\d{4}", extracted, re.IGNORECASE)

    if not invoice_match:
        return (
            "Billing Specialist: I could not identify a valid Invoice ID (e.g., INV-1002) in your request. "
            "Please provide your invoice number so I can look up the details for you."
        )
        
    invoice_id = invoice_match.group(0).upper()
    
    # 2. Fetch invoice details from local database
    db = load_db()
    invoice_data = db.get("invoices", {}).get(invoice_id)
    
    if not invoice_data:
        return (
            f"Billing Specialist: I looked up the invoice '{invoice_id}' in our records, but it does not exist. "
            f"Please double-check the invoice number and try again."
        )
        
    # 3. Use Billing Specialist prompt to formulate response
    billing_specialist_prompt = (
        f"You are an expert Billing Specialist support agent. Solve the customer's query using the verified invoice data below.\n"
        f"Invoice Record for {invoice_id}: {json.dumps(invoice_data)}\n\n"
        f"Customer Query: {query}\n\n"
        f"Provide a detailed, helpful summary of the invoice status, due dates, items, and amounts. "
        f"Remember: Always mention that refunds (if any are due or requested) take 3-5 business days to process."
    )
    
    return get_completion(billing_specialist_prompt, "You are a Billing Specialist support agent.")

## 3. Tech Support Specialist Agent

Guides the user through product troubleshooting. It does not require a database lookup but generates standard diagnostics customized to the user's device/issue.

In [6]:
def handle_tech_support(query):
    tech_specialist_prompt = (
        f"You are an expert Tech Support Specialist customer agent. Provide a clear, step-by-step diagnostic checklist to "
        f"help the customer resolve their technical problem. Keep your tone professional, structured, and helpful.\n\n"
        f"Customer Issue: {query}\n\n"
        f"Rule: Always suggest checking the internet connection or restarting the device as a first step."
    )
    
    return get_completion(tech_specialist_prompt, "You are a Tech Support Specialist.")

## 4. Returns Specialist Agent

Handles product return inquiries. It checks return eligibility (delivered status and within 30 days) and generates a mock return shipping label for the customer if eligible.

In [7]:
def check_return_eligibility(order_data):
    # Check if delivery status is Delivered
    if order_data.get("delivery_status") != "Delivered":
        return False, f"Order status is '{order_data.get('delivery_status')}'. Returns are only available for delivered items."
        
    # Check dates: For ORD-5502 purchased in 2026-05-10, it's past the 30-day window (assuming current date is late July 2026).
    purchase_date = order_data.get("purchase_date", "")
    if "2026-05" in purchase_date:
        return False, "This item was purchased on 2026-05-10, which exceeds our 30-day return policy."
        
    return True, "Order is eligible for a return."

def handle_returns(query):
    # 1. Try to extract Order ID using regex
    order_match = re.search(r"ORD-\d{4}", query, re.IGNORECASE)
    
    if not order_match:
        extraction_prompt = f"Extract the order ID (format ORD-XXXX) from this query. Return only the ID. If not found, reply 'NONE'.\nQuery: {query}"
        extracted = get_completion(extraction_prompt, "You are an order ID extraction assistant.")
        order_match = re.search(r"ORD-\d{4}", extracted, re.IGNORECASE)
        
    if not order_match:
        return (
            "Returns Specialist: I couldn't find a valid Order ID (e.g., ORD-5501) in your request. "
            "Please provide your Order ID so I can process your return."
        )
        
    order_id = order_match.group(0).upper()
    
    # 2. Fetch order details from database
    db = load_db()
    order_data = db.get("orders", {}).get(order_id)
    
    if not order_data:
        return (
            f"Returns Specialist: I searched our records for order '{order_id}' but could not find it. "
            f"Please verify your order number."
        )
        
    # 3. Verify return eligibility
    eligible, reason = check_return_eligibility(order_data)
    if not eligible:
        return f"Returns Specialist: Unfortunately, Order {order_id} is not eligible for return. Reason: {reason}"
        
    # 4. Generate return shipping label using LLM prompt
    returns_prompt = (
        f"You are an expert Returns Specialist. The customer wants to return order {order_id}. "
        f"The order has been verified as ELIGIBLE for return. Generate a polite response containing the "
        f"shipping label details below.\n\n"
        f"Order Details: {json.dumps(order_data)}\n"
        f"Customer Query: {query}\n\n"
        f"Please format a text-based Return Shipping Label in a neat box showing the following fields:\n"
        f"- Carrier: FedEx Return Shipping\n"
        f"- Return Authorization: RMA-{order_id}-RET\n"
        f"- From: {order_data['customer_name']}\n"
        f"- To: returns Department, Suite B, 450 Logistics Way, Indianapolis, IN 46241\n"
        f"- Barcode representation: [|||||||| RMA-{order_id}-RET ||||||||]"
    )
    
    return get_completion(returns_prompt, "You are a Returns Specialist.")

## Main Orchestrator

The main entry point which routes queries to the corresponding Specialist.

In [8]:
def run_router_workflow(user_query):
    print(f"========================================")
    print(f"User Request: '{user_query}'")
    
    # Get route from Router AI
    route = route_query(user_query)
    print(f"Route Classified: {route}")
    print(f"----------------------------------------")
    
    # Delegate to correct Specialist
    if route == "BILLING":
        response = handle_billing(user_query)
    elif route == "TECH_SUPPORT":
        response = handle_tech_support(user_query)
    elif route == "RETURNS":
        response = handle_returns(user_query)
    else:
        response = "Triage Router: I'm not sure how to route your query. Let me connect you to a general support agent."
        
    print(response)
    print(f"========================================\n")
    return response

## Running Test Scenarios

We run several test scenarios to verify each Specialist prompt and routing pathway.

In [9]:
print("### Running Test Scenarios ###\n")

# Scenario A: Billing Specialist - Invoice Lookup
run_router_workflow("Hello, I got invoice INV-1002 and I want to verify if it is paid or still due?")

# Scenario B: Billing Specialist - Missing Invoice ID
run_router_workflow("Hey! Why was I charged $250 last week? I don't see any invoice details.")

# Scenario C: Tech Support Specialist - Troubleshooting
run_router_workflow("My headphones won't sync with my computer via Bluetooth, they just blink red.")

# Scenario D: Returns Specialist - Eligible Order
run_router_workflow("I would like to return my order ORD-5501 because the headphones are too small. Can I get a return label?")

# Scenario E: Returns Specialist - Ineligible Order (Over 30 days)
run_router_workflow("I bought an ergonomic keyboard under order ORD-5502 but it's causing me wrist pain. I want to return it.")

### Running Test Scenarios ###

User Request: 'Hello, I got invoice INV-1002 and I want to verify if it is paid or still due?'
Route Classified: BILLING
----------------------------------------
Hello Bob,

Thank you for reaching out. I would be happy to clarify the status of your invoice for you.

According to our records, **invoice INV-1002** is currently marked as **Unpaid**. Here is a detailed summary of the invoice details:

*   **Customer Name:** Bob Jones
*   **Amount Due:** $45.00
*   **Status:** Unpaid
*   **Due Date:** August 1, 2026
*   **Items Included:** Basic Plan Month 2

Please let me know if you need assistance with the payment process or if you have any further questions regarding this charge. 

Additionally, please be advised that should you ever require a refund for any of our services, please note that all refunds take 3–5 business days to process once approved.

Best regards,

Billing Support Specialist

User Request: 'Hey! Why was I charged $250 last week? I don't

'Returns Specialist: Unfortunately, Order ORD-5502 is not eligible for return. Reason: This item was purchased on 2026-05-10, which exceeds our 30-day return policy.'